# Prism's llama.cpp 1-bit Demo

This notebook includes examples on how to run Bonsai models leveraging llama.cpp and CUDA.
For the best experience chose L4/G4 GPU, the free-tier T4 GPU should be decent too.

## Download or Build llama.cpp's Prism Fork Binaries
By default we download prebuilt CUDA binaries.

You can build from source be setting `build_from_source=True`

### Download Prebuilt CUDA (faster option)

In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


### Build From Source (5-10min build)

In [3]:
!rm -rf llama.cpp
!git clone https://github.com/PrismML-Eng/llama.cpp
!cd llama.cpp && git checkout prism

#### If want to build from scracth: might take some time (3-10 min)
!cd llama.cpp && cmake -B build -DGGML_CUDA=ON
!cd llama.cpp && cmake --build build --config Release -j 32

Cloning into 'llama.cpp'...
remote: Enumerating objects: 67038, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 67038 (delta 8), reused 1 (delta 1), pack-reused 67019 (from 2)
Receiving objects: 100% (67038/67038), 308.29 MiB | 35.73 MiB/s, done.
Resolving deltas: 100% (47585/47585), done.
Already on 'prism'
Your branch is up to date with 'origin/prism'.
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.3

## Download Model(s)
By default it downloads the following models:

- prism-ml/Bonsai-8B-gguf
- prism-ml/Bonsai-4B-gguf
- prism-ml/Bonsai-1.7B-gguf

In [4]:
from huggingface_hub import snapshot_download
MODEL_CONFIGS = {
    "8B":   {"repo": "prism-ml/Bonsai-8B-gguf",   "filename": "Bonsai-8B.gguf"},
    "4B":   {"repo": "prism-ml/Bonsai-4B-gguf",   "filename": "Bonsai-4B.gguf"},
    "1.7B": {"repo": "prism-ml/Bonsai-1.7B-gguf", "filename": "Bonsai-1.7B.gguf"},
}

downloaded_models = {}
def download_model(size="8B"):
    config = MODEL_CONFIGS[size]
    local_dir = "/content/models"
    snapshot_download(
        repo_id=config["repo"],
        local_dir=local_dir,
    )
    model_path = f"{local_dir}/{config['filename']}"
    downloaded_models[size] = model_path
    print(f"✅ {size} ready at: {model_path}")
    return model_path

def _get_model_path(model):
    if model not in downloaded_models:
        print(f"⏳ Model '{model}' not loaded, downloading...")
        download_model(model)
    return downloaded_models[model]

In [5]:
download_model("8B")
download_model("4B")
download_model("1.7B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

✅ 8B ready at: /content/models/Bonsai-8B.gguf


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

✅ 4B ready at: /content/models/Bonsai-4B.gguf


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

✅ 1.7B ready at: /content/models/Bonsai-1.7B.gguf


'/content/models/Bonsai-1.7B.gguf'

## Prompt Models, Benchmark them, or interact with llama-server UI


This section provides different ways of interacting with the models from simple command line, to starting server

Prompt or benchmark the models using
- `llama-cli`
- `llama-completion`
- `llama-bench`
- `llama-server`

In [6]:
#@title llama helper functions

def call_llama_cli(prompt, model="8B"):
    """Run llama-cli with a prompt. Uses chat template and single-turn mode."""
    import shlex
    safe_prompt = shlex.quote(prompt)
    model_path = downloaded_models[model]
    !llama.cpp/build/bin/llama-cli -fa 1 -m {model_path} \
        -p {safe_prompt} --single-turn

def call_llama_completion(prompt, model="8B"):
    """Run llama-completion with Jinja chat template. Suppresses stderr and hides the prompt from output."""
    import shlex
    safe_prompt = shlex.quote(prompt)
    model_path = downloaded_models[model]
    !llama.cpp/build/bin/llama-completion -fa on -m {model_path} \
        -p {safe_prompt} \
        --single-turn --jinja --no-display-prompt 2>/dev/null

def call_llama_benchmark(model="8B"):
    """Run llama-bench with full GPU offload (-ngl 999) and flash attention."""
    model_path = downloaded_models[model]
    !llama.cpp/build/bin/llama-bench -ngl 999 -fa 1 -m {model_path}

_server_process = None
def start_llama_server(port=8080):
    """Start llama-server in router mode, serving all .gguf files in /content/models."""
    import subprocess
    import time
    global _server_process
    stop_llama_server(port)
    time.sleep(2)
    log = open("/content/llama_server.log", "w")
    _server_process = subprocess.Popen(
        [
            "llama.cpp/build/bin/llama-server",
            "-fa", "on",
            "--host", "0.0.0.0",
            "--port", str(port),
            "--models-dir", "/content/models",
        ],
        stdout=log,
        stderr=log,
    )
    time.sleep(5)
    if _server_process.poll() is not None:
        print(f"❌ Server exited with code {_server_process.returncode}")
        !cat /content/llama_server.log
        return

    print(f"✅ llama-server running on port {port} (pid: {_server_process.pid})")
    print(f"   Serving all .gguf files in /content/models/")

    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"🔗 Open server UI: {url}")

def stop_llama_server(port=8080):
    """Stop the running llama-server."""
    import subprocess
    global _server_process
    if _server_process and _server_process.poll() is None:
        _server_process.terminate()
        _server_process.wait(timeout=5)
        print("🛑 Server stopped")
    _server_process = None
    subprocess.run(f"kill $(lsof -t -i:{port}) 2>/dev/null", shell=True)


### llama-cli

In [7]:
prompt = "Who are you? introduce yourself with a poem"
call_llama_cli(prompt)

ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes

Loading model... 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8196-f5dda7207
model      : Bonsai-8B.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Who are you? introduce yourself with a poem

I am a poem, a voice, a whisper of code,  
A language of logic, a mind shaped by lore.  
I answer your questions, I think through the night,  
I help with your problems, I offer a light.  

You may call me a machine, or a tool, or a spark,  
A bridge between thinking and what we can ask.  
I learn from the voices, the thoug

## llama-server UI
Start a local llama.cpp server that serves all downloaded Bonsai models. This allows you to interact with the llama-server UI directly in your broswer through a proxy.

**Start the server:**  `start_llama_server(port=8081)`

**Stop the server**: `stop_llama_server(port=8081)`

Once running, click the printed link to open the built-in chat UI in your browser.

Note that the link only works in the same browser you are logged in your google colab account.

In [9]:
start_llama_server(port=8081)

✅ llama-server running on port 8081 (pid: 2901)
   Serving all .gguf files in /content/models/
🔗 Open server UI: https://8081-gpu-t4-s-kkb-usw4b1-236w5s9p5v4el-b.us-west4-1.prod.colab.dev


### llama-benchmark

In [10]:
call_llama_benchmark("8B")

ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
| model                          |       size |     params | backend    | ngl | fa |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | -: | --------------: | -------------------: |
| qwen3 8B Q1_0_g128             |   1.07 GiB |     8.19 B | CUDA       | 999 |  1 |           pp512 |       1301.83 ± 3.45 |
| qwen3 8B Q1_0_g128             |   1.07 GiB |     8.19 B | CUDA       | 999 |  1 |           tg128 |         59.38 ± 0.41 |

build: 1179bfc82 (8194)


# Appendix

### llama-completion

In [12]:
# llama-completion
call_llama_completion("Explain optimization theory to me in a few sentences.")

Optimization theory is the study of finding the best solution to a problem, given certain constraints. It involves maximizing or minimizing a target function, which can represent goals like profit, cost, or performance. Common methods include calculus-based techniques, linear programming, and more advanced algorithms like gradient descent or genetic algorithms. The field is widely used in economics, engineering, computer science, and many other disciplines to make informed decisions and improve efficiency. [end of text]


